# RAG: Retrieval-Augmented Generation

En este notebook se construirá un sistema **RAG completo** paso a paso, aprenderemos a:

1. **Cargar** documentos (tu base de conocimiento)
2. **Dividirlos** en chunks
3. **Verctorizarlos** con embeddings
4. **Guardarlos** en una base de datos vectorial
5. **Buscar** información relevante para una pregunta
6. **Generar** respuestas usando un modelo de Hugging Face


---
## Instalación

Ejecuta esta celda para instalar las librerías necesarias.

In [ ]:
!pip install sentence-transformers chromadb transformers accelerate langchain langchain-community -q

---
## Sistema completo del RAG
Se va a construir un sistema RAG completo paso a paso que contrendrá:

1. **Chunking**: Dividir documentos en trozos manejables
2. **Embeddings**: Convertir texto en vectores para búsqueda semántica
3. **Vector Store**: Almacenar y buscar eficientemente con ChromaDB
4. **Retrieval**: Encontrar información relevante para cada pregunta
5. **Generación**: Producir respuestas usando modelos de Hugging Face

---
## Datos de ejemplo

Vamos a crear una pequeña "base de conocimiento" sobre lenguajes de programación. En un caso real, esto vendría de PDFs, documentos de tu empresa, etc.

In [ ]:
#Base de conocimiento de ejemplo
DOCUMENTOS = {
    "python": """
    Python es un lenguaje de programación de alto nivel creado por Guido van Rossum en 1991.
    Es conocido por su sintaxis clara y legible, lo que lo hace ideal para principiantes.
    Python es interpretado, lo que significa que no necesita compilación.
    Se usa ampliamente en ciencia de datos, inteligencia artificial, desarrollo web y automatización.
    Las principales librerías de Python para IA son TensorFlow, PyTorch, scikit-learn y Hugging Face.
    Python tiene una comunidad muy grande y activa, con miles de paquetes disponibles en PyPI.
    El Zen de Python establece principios como "Simple es mejor que complejo" y "La legibilidad cuenta".
    """,
    
    "javascript": """
    JavaScript fue creado por Brendan Eich en 1995 en solo 10 días mientras trabajaba en Netscape.
    Originalmente se llamaba Mocha, luego LiveScript, y finalmente JavaScript.
    JavaScript es el lenguaje principal para desarrollo web frontend y ahora también backend con Node.js.
    Es un lenguaje interpretado que se ejecuta en el navegador del usuario.
    Las principales frameworks de JavaScript son React, Vue, Angular para frontend y Express, NestJS para backend.
    JavaScript usa tipado dinámico, aunque TypeScript añade tipado estático opcional.
    NPM (Node Package Manager) es el gestor de paquetes más grande del mundo.
    """,
    
    "rust": """
    Rust es un lenguaje de programación de sistemas creado por Mozilla, lanzado en 2010.
    Su principal característica es la seguridad de memoria sin necesidad de recolector de basura.
    Rust usa un sistema de ownership (propiedad) único que previene errores de memoria en tiempo de compilación.
    Es extremadamente rápido, comparable a C y C++, pero mucho más seguro.
    Se usa para sistemas operativos, navegadores web (Firefox), herramientas de línea de comandos y WebAssembly.
    Rust ha sido votado como el lenguaje "más amado" en Stack Overflow durante varios años consecutivos.
    Cargo es el gestor de paquetes y build system de Rust.
    """,
    
    "go": """
    Go (también llamado Golang) fue creado por Google en 2009 por Robert Griesemer, Rob Pike y Ken Thompson.
    Fue diseñado para ser simple, eficiente y fácil de aprender.
    Go es un lenguaje compilado con recolector de basura y soporte nativo para concurrencia.
    Las goroutines son hilos ligeros que permiten manejar miles de tareas concurrentes fácilmente.
    Se usa principalmente para servicios backend, microservicios, herramientas de DevOps y cloud computing.
    Docker y Kubernetes están escritos en Go.
    Go tiene un sistema de tipos estático pero con inferencia de tipos para mayor comodidad.
    """
}

print(f"Base de conocimiento cargada: {len(DOCUMENTOS)} documentos")
for nombre, contenido in DOCUMENTOS.items():
    print(f"   - {nombre}: {len(contenido)} caracteres")

---
# Ejercicio 1: Chunking (Dividir documentos)

Los documentos largos deben dividirse en "chunks" (trozos) más pequeños porque:
- Los modelos tienen límite de tokens
- Chunks más pequeños = búsqueda más precisa

### Ejemplo: Chunking manual

In [ ]:
#Ejemplo: Chunking simple por número de caracteres
def chunking_simple(texto, chunk_size=200, overlap=50):
    """
    Divide un texto en chunks de tamaño fijo con solapamiento.
    
    Args:
        texto: El texto a dividir
        chunk_size: Tamaño máximo de cada chunk
        overlap: Caracteres de solapamiento entre chunks
    """
    chunks = []
    inicio = 0
    
    while inicio < len(texto):
        fin = inicio + chunk_size
        chunk = texto[inicio:fin]
        chunks.append(chunk.strip())
        inicio = fin - overlap  #Retrocedemos para el solapamiento
    
    return chunks

#Probar con el documento de Python
chunks_python = chunking_simple(DOCUMENTOS["python"], chunk_size=200, overlap=30)

print(f"Documento original: {len(DOCUMENTOS['python'])} caracteres")
print(f"Chunks generados: {len(chunks_python)}\n")

for i, chunk in enumerate(chunks_python):
    print(f"--- Chunk {i+1} ({len(chunk)} chars) ---")
    print(chunk)
    print()

### Problema del chunking simple

El chunking por caracteres puede cortar frases a la mitad. Es mejor usar **chunking inteligente** que respete los límites de las frases.

### Ejemplo: Chunking con LangChain

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

#RecursiveCharacterTextSplitter intenta dividir por párrafos, luego frases, luego palabras
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=30,
    separators=["\n\n", "\n", ". ", " ", ""]  #Orden de preferencia para dividir
)

chunks_python_smart = splitter.split_text(DOCUMENTOS["python"])

print(f"Chunks generados (inteligente): {len(chunks_python_smart)}\n")

for i, chunk in enumerate(chunks_python_smart):
    print(f"--- Chunk {i+1} ({len(chunk)} chars) ---")
    print(chunk)
    print()

### Experimenta con el chunking

**Qué hay que hacer:**
1. Divide TODOS los documentos en chunks usando `RecursiveCharacterTextSplitter`
2. Guarda también el nombre del documento de origen (metadatos)
3. Experimenta con diferentes valores de `chunk_size` (prueba 150, 300, 500)

**Pregunta:** ¿Qué chunk_size crees que funcionará mejor para búsqueda? ¿Por qué?

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

#TODO: Crea un splitter con el chunk_size que prefieras
splitter = RecursiveCharacterTextSplitter(
    chunk_size=...,  # TODO: elige un tamaño
    chunk_overlap=30,
)

#TODO: Divide TODOS los documentos y guarda el origen
todos_los_chunks = []  # Lista de {"texto": ..., "origen": ...}

for nombre_doc, contenido in DOCUMENTOS.items():
    #TODO: Dividir el documento en chunks
    chunks = ...
    
    #TODO: Guardar cada chunk con su origen
    for chunk in chunks:
        todos_los_chunks.append({
            "texto": ...,
            "origen": ...
        })

#Mostrar resultados
print(f"Total de chunks: {len(todos_los_chunks)}\n")
for i, chunk_info in enumerate(todos_los_chunks[:5]):  #Mostrar primeros 5
    print(f"Chunk {i+1} (de {chunk_info['origen']}):")
    print(f"  {chunk_info['texto'][:100]}...\n")

---
# Ejercicio 2: Embeddings (Vectorizar texto)

Los embeddings convierten texto en vectores numéricos. Textos con significado similar tendrán vectores cercanos.

### Ejemplo: Crear embeddings con sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

#Cargar modelo de embeddings (multilingual para español)
modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

#Ejemplo: Crear embedding de un texto
texto_ejemplo = "Python es un lenguaje de programación"
embedding = modelo_embeddings.encode(texto_ejemplo)

print(f"Texto: {texto_ejemplo}")
print(f"Embedding shape: {embedding.shape}")
print(f"Primeros 10 valores: {embedding[:10]}")

### Ejemplo: Similitud entre textos

Podemos calcular qué tan similares son dos textos comparando sus embeddings.

In [ ]:
from sentence_transformers import util

#Tres textos para comparar
textos = [
    "Python es un lenguaje de programación",
    "Java es un lenguaje de programación",
    "Me gusta comer pizza los viernes"
]

#Crear embeddings
embeddings = modelo_embeddings.encode(textos)

#Calcular similitud coseno entre todos los pares
print("Matriz de similitud:\n")
for i, texto_i in enumerate(textos):
    for j, texto_j in enumerate(textos):
        similitud = util.cos_sim(embeddings[i], embeddings[j]).item()
        if i <= j:
            print(f"'{texto_i[:30]}...' vs '{texto_j[:30]}...'")
            print(f"   Similitud: {similitud:.4f}\n")

### Embeddings de tus chunks

**QUé hay que hacer:**
1. Crea embeddings para todos los chunks del ejercicio anterior
2. Prueba a buscar el chunk más similar a una pregunta
3. Experimenta con diferentes preguntas

In [ ]:
from sentence_transformers import SentenceTransformer, util

#Usar el modelo multilingual
modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

#TODO: Crear embeddings para todos los chunks
#Suponiendo que tienes todos_los_chunks del ejercicio anterior

#Extraer solo los textos
textos_chunks = [chunk["texto"] for chunk in todos_los_chunks]

#TODO: Crear embeddings de todos los chunks
embeddings_chunks = ...  # modelo_embeddings.encode(...)

print(f"Embeddings creados: {embeddings_chunks.shape}")

In [ ]:
#TODO: Función para buscar chunks similares a una pregunta

def buscar_chunks_similares(pregunta, top_k=3):
    """
    Busca los chunks más similares a una pregunta.
    
    Args:
        pregunta: La pregunta del usuario
        top_k: Número de chunks a devolver
    
    Returns:
        Lista de los chunks más similares con su score
    """
    #TODO: Crear embedding de la pregunta
    embedding_pregunta = ...
    
    #TODO: Calcular similitud con todos los chunks
    similitudes = util.cos_sim(embedding_pregunta, embeddings_chunks)[0]
    
    #TODO: Obtener los top_k más similares
    # Pista: usa torch.topk() o np.argsort()
    indices_top = ...
    
    #Devolver resultados
    resultados = []
    for idx in indices_top:
        resultados.append({
            "texto": todos_los_chunks[idx]["texto"],
            "origen": todos_los_chunks[idx]["origen"],
            "score": similitudes[idx].item()
        })
    
    return resultados

#Probar
pregunta = "¿Quién creó Python?"
resultados = buscar_chunks_similares(pregunta, top_k=3)

print(f"Pregunta: {pregunta}\n")
print("Chunks más relevantes:")
for i, r in enumerate(resultados):
    print(f"\n{i+1}. (score: {r['score']:.4f}) [{r['origen']}]")
    print(f"   {r['texto'][:150]}...")

---
# Ejercicio 3: Vector Store (ChromaDB)

En lugar de manejar embeddings manualmente, usamos una **base de datos vectorial** que:
- Almacena los embeddings eficientemente
- Permite búsquedas rápidas por similitud
- Guarda metadatos junto con los documentos

### Ejemplo: ChromaDB básico

In [ ]:
import chromadb

#Crear cliente de ChromaDB (en memoria)
client = chromadb.Client()

#Crear una colección (como una tabla)
coleccion = client.create_collection(
    name="ejemplo_basico",
    metadata={"description": "Mi primera colección"}
)

#Añadir documentos (ChromaDB genera embeddings automáticamente con su modelo por defecto)
coleccion.add(
    documents=["Python es genial", "JavaScript es popular", "Rust es rápido"],
    ids=["doc1", "doc2", "doc3"],
    metadatas=[{"tema": "python"}, {"tema": "javascript"}, {"tema": "rust"}]
)

print(f"Documentos en la colección: {coleccion.count()}")

In [ ]:
#Buscar documentos similares
resultados = coleccion.query(
    query_texts=["¿Qué lenguaje es más veloz?"],
    n_results=2
)

print("Resultados de búsqueda:")
print(f"Documentos: {resultados['documents']}")
print(f"Distancias: {resultados['distances']}")
print(f"Metadatos: {resultados['metadatas']}")

### Crea tu Vector Store

**Qué hay que hacer:**
1. Crea una colección en ChromaDB con todos tus chunks
2. Incluye metadatos (el documento de origen)
3. Haz búsquedas con diferentes preguntas
4. Filtra resultados por metadatos (solo de un documento específico)

In [ ]:
import chromadb

#Crear cliente (nuevo, limpio)
client = chromadb.Client()

#TODO: Crear colección para tu base de conocimiento
mi_coleccion = client.create_collection(
    name="base_conocimiento",
)

#TODO: Añadir todos los chunks con sus metadatos
#Necesitas: documents (lista de textos), ids (lista de IDs únicos), metadatas (lista de dicts)

documentos = ...
ids = ...
metadatas = ...

mi_coleccion.add(
    documents=documentos,
    ids=ids,
    metadatas=metadatas
)

print(f"Añadidos {mi_coleccion.count()} chunks a la colección")

In [ ]:
#TODO: Función para buscar en tu colección

def buscar_en_coleccion(pregunta, n_resultados=3, filtro_origen=None):
    """
    Busca chunks relevantes en la colección.
    
    Args:
        pregunta: La pregunta del usuario
        n_resultados: Número de resultados
        filtro_origen: Opcional, filtrar por documento de origen
    """
    #Construir filtro si se especifica
    where_filter = None
    if filtro_origen:
        where_filter = {"origen": filtro_origen}
    
    #TODO: Hacer la búsqueda
    resultados = mi_coleccion.query(
        query_texts=[pregunta],
        n_results=n_resultados,
        where=where_filter
    )
    
    return resultados

#Probar búsqueda general
print("=== Búsqueda general ===")
resultados = buscar_en_coleccion("¿Qué lenguaje es bueno para IA?")
for doc, meta in zip(resultados['documents'][0], resultados['metadatas'][0]):
    print(f"\n[{meta['origen']}]: {doc[:100]}...")

#TODO: Probar búsqueda con filtro (solo en un documento específico)
print("\n=== Búsqueda filtrada (solo Python) ===")

---
# Ejercicio 4: Modelo de Generación

Ahora necesitamos un modelo que genere respuestas basadas en el contexto recuperado.

### Ejemplo: Pipeline de generación de texto

In [ ]:
from transformers import pipeline

#Cargar modelo de generación
#gpt2 es pequeño y rápido, pero no muy bueno para seguir instrucciones
generador = pipeline(
    "text-generation",
    model="gpt2",
    max_new_tokens=100,
    pad_token_id=50256  # Evitar warning
)

#Probar generación simple
prompt = "Python es un lenguaje de programación que"
respuesta = generador(prompt)[0]['generated_text']

print(f"Prompt: {prompt}")
print(f"\nRespuesta generada:\n{respuesta}")

### Modelo mejor para instrucciones: FLAN-T5

FLAN-T5 está entrenado para seguir instrucciones, mejor para RAG.

In [ ]:
from transformers import pipeline

#FLAN-T5 es mejor para seguir instrucciones
generador_flan = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    max_new_tokens=100
)

#Probar con una instrucción
prompt = """Responde la pregunta basándote en el contexto.

Contexto: Python fue creado por Guido van Rossum en 1991. Es un lenguaje interpretado.

Pregunta: ¿Quién creó Python?

Respuesta:"""

respuesta = generador_flan(prompt)[0]['generated_text']
print(f"Respuesta: {respuesta}")

### Elige y configura tu modelo

**Qué hay que hacer:**
1. Prueba al menos 2 modelos diferentes
2. Experimenta con los parámetros (max_new_tokens, temperature, etc.)
3. Crea una función que reciba contexto + pregunta y devuelva respuesta

**Modelos sugeridos:**
- `gpt2` - Pequeño, rápido, no muy bueno con instrucciones
- `google/flan-t5-base` - Bueno con instrucciones, tamaño medio
- `google/flan-t5-small` - Más rápido, menos calidad
- `microsoft/DialoGPT-medium` - Entrenado para diálogo

In [ ]:
from transformers import pipeline

#TODO: Elige tu modelo
MODELO_ELEGIDO = "google/flan-t5-base"  # Cambia esto

#TODO: Determina el tipo de pipeline según el modelo
#- "text-generation" para GPT-2, DialoGPT, etc.
#- "text2text-generation" para FLAN-T5, T5, etc.
TIPO_PIPELINE = ...  

#Crear el generador
mi_generador = pipeline(
    TIPO_PIPELINE,
    model=MODELO_ELEGIDO,
    max_new_tokens=150
)

print(f"Modelo cargado: {MODELO_ELEGIDO}")

In [ ]:
#TODO: Función para generar respuesta dado contexto y pregunta

def generar_respuesta(contexto, pregunta):
    """
    Genera una respuesta basada en el contexto.
    
    Args:
        contexto: Información relevante (chunks recuperados)
        pregunta: La pregunta del usuario
    
    Returns:
        Respuesta generada
    """
    #TODO: Construir el prompt
    #El formato del prompt es muy importante para buenos resultados
    prompt = f"""...
    
    """
    
    #TODO: Generar respuesta
    respuesta = ...
    
    return respuesta

#Probar
contexto_prueba = "Python fue creado por Guido van Rossum en 1991. Es un lenguaje interpretado de alto nivel."
pregunta_prueba = "¿En qué año se creó Python?"

respuesta = generar_respuesta(contexto_prueba, pregunta_prueba)
print(f"Pregunta: {pregunta_prueba}")
print(f"Respuesta: {respuesta}")

---
# Ejercicio 5: RAG Completo

Ahora juntamos todo: Vector Store + Retrieval + Generación.

### Construye tu sistema RAG completo

In [ ]:
#Sistema RAG completo

class SistemaRAG:
    def __init__(self, coleccion, generador, n_chunks=3):
        """
        Inicializa el sistema RAG.
        
        Args:
            coleccion: Colección de ChromaDB con los documentos
            generador: Pipeline de generación de Hugging Face
            n_chunks: Número de chunks a recuperar
        """
        self.coleccion = coleccion
        self.generador = generador
        self.n_chunks = n_chunks
    
    def recuperar_contexto(self, pregunta):
        """
        Recupera los chunks más relevantes para la pregunta.
        """
        #TODO: Buscar en la colección
        resultados = ...
        
        #TODO: Extraer los textos y unirlos
        chunks = ...
        contexto = "\n".join(chunks)
        
        return contexto, resultados
    
    def generar(self, contexto, pregunta):
        """
        Genera una respuesta usando el modelo.
        """
        #TODO: Construir prompt y generar respuesta
        prompt = f"""Responde la pregunta basándote ÚNICAMENTE en el contexto proporcionado.
Si la información no está en el contexto, di "No tengo información sobre eso".

Contexto:
{contexto}

Pregunta: {pregunta}

Respuesta:"""
        
        respuesta = self.generador(prompt)[0]['generated_text']
        
        #Para text2text-generation, la respuesta es directa
        #Para text-generation, hay que extraer solo la parte nueva
        return respuesta
    
    def preguntar(self, pregunta, verbose=True):
        """
        Pipeline completo: recuperar + generar.
        """
        #1. Recuperar contexto
        contexto, resultados = self.recuperar_contexto(pregunta)
        
        if verbose:
            print(f"📚 Chunks recuperados:")
            for doc, meta in zip(resultados['documents'][0], resultados['metadatas'][0]):
                print(f"   [{meta.get('origen', 'unknown')}]: {doc[:80]}...")
            print()
        
        #2. Generar respuesta
        respuesta = self.generar(contexto, pregunta)
        
        return respuesta

In [ ]:
#TODO: Crear tu sistema RAG

#Asegúrate de tener:
#- mi_coleccion: tu colección de ChromaDB con todos los chunks
#- mi_generador: tu pipeline de generación

rag = SistemaRAG(
    coleccion=mi_coleccion,
    generador=mi_generador,
    n_chunks=3
)

print("Sistema RAG inicializado")

In [ ]:
#TODO: Probar tu sistema RAG con varias preguntas

preguntas_test = [
    "¿Quién creó Python?",
    "¿Para qué se usa JavaScript?",
    "¿Por qué Rust es seguro?",
    "¿Qué está escrito en Go?",
    "¿Qué lenguaje es mejor para IA?",
]

for pregunta in preguntas_test:
    print(f"\n{'='*60}")
    print(f"Pregunta: {pregunta}")
    print(f"{'='*60}")
    
    respuesta = rag.preguntar(pregunta)
    print(f"\nRespuesta: {respuesta}")

---
# Ejercicio 6: Mejora tu RAG

Tu RAG básico funciona, pero puede mejorarse. Implementa al menos 2 de estas mejoras:

### Mejoras posibles:

1. **Mejor prompt**: Experimenta con diferentes formatos de prompt
2. **Más contexto**: Ajusta n_chunks según la pregunta
3. **Filtrado por relevancia**: Solo usa chunks con score > umbral
4. **Mostrar fuentes**: Incluye de qué documento vino cada parte
5. **Historial**: Mantén contexto de preguntas anteriores
6. **Modelo diferente**: Prueba otro modelo de generación

In [ ]:
#TODO: Implementa tu versión mejorada de SistemaRAG

class SistemaRAGMejorado:
    def __init__(self, coleccion, generador, n_chunks=3, umbral_relevancia=None):
        self.coleccion = coleccion
        self.generador = generador
        self.n_chunks = n_chunks
        self.umbral_relevancia = umbral_relevancia
        self.historial = []  # Para almacenar preguntas anteriores
    
    def recuperar_contexto(self, pregunta):
        #TODO: Implementar recuperación mejorada
        #- Filtrar por umbral de relevancia
        #- Devolver también metadatos para citar fuentes
        pass
    
    def generar(self, contexto, pregunta, fuentes):
        #TODO: Implementar generación mejorada
        #- Mejor prompt
        #- Incluir historial si es relevante
        pass
    
    def preguntar(self, pregunta):
        #TODO: Pipeline mejorado
        #- Mostrar fuentes en la respuesta
        #- Guardar en historial
        pass

#Probar tu versión mejorada

---
# Ejercicio EXTRA: Añade tus propios documentos

**Qué hay que hacer:**
1. Crea documentos sobre un tema que te interese (o usa PDFs/textos reales)
2. Procésalos con tu pipeline RAG
3. Haz preguntas sobre tu nueva base de conocimiento

In [ ]:
#TODO: Añade tus propios documentos

MIS_DOCUMENTOS = {
    "tema1": """
    Tu contenido aquí...
    """,
    "tema2": """
    Más contenido...
    """
}

#Procesa y añade a una nueva colección